# T4 DSA+SEA+Snapshot Cross-Dataset Runner

This notebook assumes the preprocessed Drive files already exist and only loads them for training.

Required legacy sfreq100 files:
- `/content/drive/MyDrive/MI_loso_project/project/crossdata/preprocessed_sfreq100/cho2017.npz`
- `/content/drive/MyDrive/MI_loso_project/project/crossdata/preprocessed_sfreq100/lee2019.npz`

Expected shapes from the existing cross-dataset matrix: Cho2017 `(10520, 64, 201)`, Lee2019 `(10800, 62, 201)`. Do not use a Lee2019 `(5400, 62, 201)` one-session export.

Runtime: select `T4 GPU` before running.

## 1. Runtime And Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, platform
print('Python:', sys.version)
print('Platform:', platform.platform())
!nvidia-smi

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
Sun Jun 28 12:17:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0

## 2. Clone Or Update Repository

In [2]:
import os

REPO_URL = 'https://github.com/heegyukim4043/MI_loso_project.git'
REPO_DIR = '/content/MI_loso_project'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}/project/crossdata/models
!git rev-parse --short HEAD

/content/MI_loso_project
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 4 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.43 KiB | 734.00 KiB/s, done.
From https://github.com/heegyukim4043/MI_loso_project
   865efdf..a8a03e0  master     -> origin/master
Updating 865efdf..a8a03e0
Fast-forward
 project/colab_dsa_sea_snapshot_runner.ipynb | 89 +++++++++++++++--------------
 1 file changed, 45 insertions(+), 44 deletions(-)
/content/MI_loso_project/project/crossdata/models
a8a03e0


## 3. Set Data Path And Verify Files

No MOABB download or preprocessing is run here. The notebook stops if the two `.npz` files are missing or malformed.

In [ ]:
import os
import numpy as np

os.environ['MI_PREPROCESSED_DIR'] = '/content/drive/MyDrive/MI_loso_project/project/crossdata/preprocessed_sfreq100'
os.environ['MI_BACKUP_DIR'] = '/content/drive/MyDrive/MI_loso_project/colab_results/dsa_sea_snapshot_20260628v2'
os.environ['MI_N_TIMES'] = '201'

prep = os.environ['MI_PREPROCESSED_DIR']
print('MI_PREPROCESSED_DIR =', prep)
print('MI_BACKUP_DIR       =', os.environ['MI_BACKUP_DIR'])
!ls -lh "$MI_PREPROCESSED_DIR"

# ── 검증 ─────────────────────────────────────────────────────────────────────
REQUIRED = {
    'cho2017': {
        'shape': (10520, 64, 201),
        'n_subjects': 52,
        'trials_per_subject': None,   # 200 or 240 mixed (정상)
        'sfreq': 100.0,
    },
    'lee2019': {
        'shape': (10800, 62, 201),    # 54 subjects × 200 trials = 10800
        'n_subjects': 54,
        'trials_per_subject': 200,    # 양 세션 = 200
        'sfreq': 100.0,
    },
}

for name, spec in REQUIRED.items():
    path = os.path.join(prep, f'{name}.npz')
    if not os.path.exists(path):
        raise FileNotFoundError(
            f'{path}\n'
            f'Available: {os.listdir(prep)}'
        )

    d        = np.load(path, allow_pickle=True)
    X        = d['X']
    y        = d['y']
    subjects = d['subjects']
    ch_names = d['ch_names']
    sfreq    = float(d['sfreq'])

    unique_subjs, counts = np.unique(subjects, return_counts=True)
    trials_set = sorted(set(counts.tolist()))

    print(f'\n{name}:')
    print(f'  X={X.shape}  subjects={len(unique_subjs)}  '
          f'ch={len(ch_names)}  sfreq={sfreq}')
    print(f'  trials/subject: {trials_set}')
    print(f'  labels: {np.unique(y, return_counts=True)}')

    # shape 검증
    assert tuple(X.shape) == spec['shape'], \
        f'{name}: X={X.shape} ≠ expected {spec["shape"]}'
    assert abs(sfreq - spec['sfreq']) < 1e-6, \
        f'{name}: sfreq={sfreq} ≠ {spec["sfreq"]}'
    assert len(unique_subjs) == spec['n_subjects'], \
        f'{name}: {len(unique_subjs)} subjects ≠ {spec["n_subjects"]}'
    assert len(y) == X.shape[0] and len(subjects) == X.shape[0], \
        f'{name}: length mismatch'

    # trials/subject 검증
    if spec['trials_per_subject'] is not None:
        wrong = [(int(s), int(c)) for s, c in zip(unique_subjs, counts)
                 if c != spec['trials_per_subject']]
        assert not wrong, (
            f'{name}: subjects {wrong[:5]} have unexpected trial counts '
            f'(expected {spec["trials_per_subject"]}). '
            f'Full distribution: {trials_set}'
        )

    print(f'  ✅ {name} OK')

print('\n✅ All datasets validated — 기존 프로토콜 일치 (Lee2019 양 세션 200 trials/subject)')

MI_PREPROCESSED_DIR = /content/drive/MyDrive/MI_loso_project/project/crossdata/preprocessed_sfreq100
MI_BACKUP_DIR       = /content/drive/MyDrive/MI_loso_project/colab_results/dsa_sea_snapshot_20260628
total 718M
-rw------- 1 root root 480M Jun 27 18:43 cho2017.npz
-rw------- 1 root root 239M Jun 27 18:12 lee2019.npz

cho2017:
  X=(10520, 64, 201)  subjects=52  ch=64  sfreq=100.0
  trials/subject: [200, 240]
  labels: (array([0, 1]), array([5260, 5260]))
  ✅ cho2017 OK

lee2019:
  X=(5400, 62, 201)  subjects=54  ch=62  sfreq=100.0
  trials/subject: [100]
  labels: (array([0, 1]), array([2700, 2700]))


ValueError: lee2019: subjects [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)] have [100, 100, 100, 100, 100] trials — expected 200 (both sessions). 
Found trial counts: [100]
This NPZ appears to contain session 1 only. Re-export with sessions=(1,2) or use preprocessed_raw_unified instead.

## 4. Dependency Sanity Check

Colab already includes the core stack. This cell only checks imports.

In [ ]:
import torch, scipy, sklearn
print('torch', torch.__version__)
print('cuda', torch.cuda.is_available())
print('scipy', scipy.__version__)
print('sklearn', sklearn.__version__)
!python -m py_compile manage_colab_dsa_sea_snapshot_20260628.py cross_dataset.py

## 5. Run CSPNet Cho -> Lee

Set `FORCE=True` only when rerunning after a failed or interrupted attempt.

In [ ]:


!python manage_colab_dsa_sea_snapshot_20260628.py \
  --models cspnet \
  --direction cho2lee \
  --preprocessed_dir "$MI_PREPROCESSED_DIR" \
  --backup_dir "$MI_BACKUP_DIR" \
  --force

## 6. Save Current Results To Drive

In [ ]:
SAVE_DIR = os.environ['MI_BACKUP_DIR']
!mkdir -p "$SAVE_DIR"
!cp -v /content/MI_loso_project/project/crossdata/colab_dsa_sea_snapshot_20260628.md "$SAVE_DIR"/ 2>/dev/null || true
!cp -v /content/MI_loso_project/project/crossdata/results/loso_results_20260628_colab_dsa_sea_snapshot_*_cross_*.csv "$SAVE_DIR"/ 2>/dev/null || true
!ls -lh "$SAVE_DIR"

## 7. Run CSPNet Lee -> Cho

In [ ]:
!python manage_colab_dsa_sea_snapshot_20260628.py \
  --models cspnet \
  --direction lee2cho \
  --preprocessed_dir "$MI_PREPROCESSED_DIR" \
  --backup_dir "$MI_BACKUP_DIR" \
  --force

## 8. Optional: EEGNet Both Directions

In [ ]:
!python manage_colab_dsa_sea_snapshot_20260628.py \
  --models eegnet \
  --direction both \
  --preprocessed_dir "$MI_PREPROCESSED_DIR" \
  --backup_dir "$MI_BACKUP_DIR" \
  --force

## 9. Optional: Conformer Both Directions

In [ ]:
!python manage_colab_dsa_sea_snapshot_20260628.py \
  --models conformer \
  --direction both \
  --preprocessed_dir "$MI_PREPROCESSED_DIR" \
  --backup_dir "$MI_BACKUP_DIR" \
  --force

## 10. Logs And Summary

In [ ]:
%cd /content/MI_loso_project/project/crossdata
!cat colab_dsa_sea_snapshot_20260628.md || true
!ls -lh results/runs/*dsa_sea_snapshot*.log 2>/dev/null || true
!tail -n 80 results/runs/*dsa_sea_snapshot*.log 2>/dev/null || true
!ls -lh results/loso_results_20260628_colab_dsa_sea_snapshot_*_cross_*.csv 2>/dev/null || true

## 11. Push Verified Results To GitHub

Run this only after checking the backup folder contains the expected summary and CSV files. This step is intentionally manual and asks for a GitHub token at runtime.

In [ ]:
from getpass import getpass
import os

if not os.environ.get('GITHUB_TOKEN'):
    os.environ['GITHUB_TOKEN'] = getpass('GitHub token with contents:write permission: ')

%cd /content/MI_loso_project
!git pull --ff-only origin master
!mkdir -p project/crossdata/results
!cp -v "$MI_BACKUP_DIR"/colab_dsa_sea_snapshot_20260628.md project/crossdata/colab_dsa_sea_snapshot_20260628.md
!cp -v "$MI_BACKUP_DIR"/loso_results_20260628_colab_dsa_sea_snapshot_*.csv project/crossdata/results/
!git status --short
!git config user.name "heegyukim4043"
!git config user.email "55726335+heegyukim4043@users.noreply.github.com"
!git add project/crossdata/colab_dsa_sea_snapshot_20260628.md project/crossdata/results/loso_results_20260628_colab_dsa_sea_snapshot_*.csv
!git commit -m "Add Colab DSA SEA Snapshot results" || true
!git -c http.extraHeader="AUTHORIZATION: bearer $GITHUB_TOKEN" push origin master